In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load


# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import pandas as pd
import cv2
import string
import sys
import os
import numpy as np
import contextlib
import random
from copy import deepcopy
from concurrent.futures import ProcessPoolExecutor, as_completed
import json
import subprocess

sys.path.append("./tetris_a/src")
from game import Game

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All"
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

pygame 2.6.1 (SDL 2.28.4, Python 3.12.6)
Hello from the pygame community. https://www.pygame.org/contribute.html


# 1. Frame the problem
Using the customer description, Define the problem your trying to solve in your own words (remember this is not technial but must be specific so the customer understands the project

We want to build an automated Tetris player than will play Tetris in an headless mode environment (which allows the bot to play without waiting for regular game delay). The goal is to maximize how long the bot lasts before losing. To solve the problem, we will use a machine learning model adapted to the task to train our bot player.

# 2. Get the Data 
Define how you recieved the data (provided, gathered..)

We were given a link to an implementation of tetris: https://gitlab.com/yukiman/tetris_a. This repository also allows the game to run in headless mode, and provides several bot implementations.

# 3. Explore the Data
Gain insights into the data you have from step 2, making sure to identify any bias

In [2]:
def run_strategy(strategy):
    os.system("clear")
    result = subprocess.run(
        ["python", "tetris_a/src/main.py", strategy, "--nodisplay"],
        capture_output=True,
        text=True
    )
    all_output = result.stderr + result.stdout
    return all_output.count('\n') - 1

trials = 10
mcts_average = sum([run_strategy("mcts") for i in range(trials)]) / trials
random_average = sum([run_strategy("randomChoice") for i in range(trials)]) / trials
genetic_average = sum([run_strategy("genetic") for i in range(trials)]) / trials
greedy_average = sum([run_strategy("greedy") for i in range(trials)]) / trials
print(f"Monte Carlo: {mcts_average}")
print(f"Random: {random_average}")
print(f"Genetic: {genetic_average}")
print(f"Greedy: {greedy_average}")

Monte Carlo: 11.5
Random: 1.0
Genetic: 18.3
Greedy: 6028.1


We used the existing models from the GitHub repository to get a sense of how strong our bot should be. The genetic model performs very poorly, and so it has likely not been trained yet. The Monte Carlo Tree Search also performs poorly, however, and I do not know why this is the case. The greedy algorithm is the only viable bot player, and over 10 simulations it survived for an average of 9026.3 moves. We will take the greedy algorithm as a benchmark score for the model we will train. However, we should expect that our genetic model will perform much better than the simple greedy algorithm since the genetic algorithm will actually "learn".

# 4.Prepare the Data


Apply any data transformations and explain what and why


Our genetic model will learn as it plays. The game itself is the data, and so no data transformations are necessary. Here we will outline how our bot will train:

We will select certain genes for our model, and each bot will have assigned weights for each gene, which corresponds to some feature of the game board. The bot will then pick the move that minimizes the weighted sum, and we will use an elitist survival model, along with reproduction and mutation to create new bots in the next generation. The genes we will use are number of lines cleared, sum of column heights, maximum column height, number of empty cells with a filled square above, sum of absolute height differences between columns for now (lines, total height, bumpiness, and holes for now).

# 5. Model the data
Using selected ML models, experment with your choices and describe your findings. Finish by selecting a Model to continue with


In [2]:
def column_heights(grid):
    return np.where(grid.any(axis=0), grid.shape[0] - np.argmax(grid[::-1, :], axis=0), 0)

def total_height(grid, column_heights):
    return np.sum(column_heights)

def count_holes(grid):
    return np.sum((np.cumsum(grid, axis=0) > 0) & (~grid))

def maximum_height(grid, column_heights):
    return np.max(column_heights)

def bumpiness(grid, column_heights):
    return np.sum(np.abs(np.diff(column_heights)))

def wells(grid, column_heights):
    h = grid.shape[0]
    padded = np.concatenate([[h], column_heights, [h]])
    return np.sum(np.maximum(0, np.minimum(padded[:-2], padded[2:]) - padded[1:-1]))

def row_transitions(grid):
    padded = np.pad(grid, ((0, 0), (1, 1)), constant_values=1)
    return np.sum(np.sum(padded[:, :-1] != padded[:, 1:], axis=1))

def column_transitions(grid):
    padded = np.pad(grid, ((0, 1), (0, 0)), constant_values=1)
    return np.sum(np.sum(padded[:-1, :] != padded[1:, :], axis=0))

def total_filled(grid):
    return np.sum(grid)

def valuation(board, chromosome):
    sim_board = deepcopy(board)
    rows_cleared = sim_board.clear_rows()
    grid = np.array(sim_board.board, dtype=bool)
    heights = column_heights(grid)
    features = np.array([
        total_height(grid, heights),
        maximum_height(grid, heights),
        count_holes(grid),
        bumpiness(grid, heights),
        rows_cleared,
        wells(grid, heights),
        row_transitions(grid),
        column_transitions(grid),
        total_filled(grid)
    ])
    return np.dot(features, chromosome)

We define attributes that can quantify a board, and finally define a valuation function. The valuation function takes in the chromosome and outputs the model's evaluation of that board position.

In [3]:
class Genetic_AI:
    def __init__(self, chromosome=None):
        if chromosome is None:
            self.chromosome = np.random.uniform(-1, 1, 4)
        else:
            self.chromosome = np.array(chromosome)
    def evaluate_board(self, board):
        return valuation(board, self.chromosome)
    def get_possible_moves(self, board, piece):
        seen = set()
        rotations = []
        while True:
            body = tuple(sorted(piece.body))
            if body in seen:
                break
            seen.add(body)
            rotations.append(piece)
            piece = piece.get_next_rotation()
        moves_list = []
        for rotation in rotations:
            max_x = board.width - len(rotation.skirt)
            moves_list.extend([(x, rotation) for x in np.arange(max_x + 1)])
        moves = np.array(moves_list, dtype=object)
        return moves
    def get_best_move(self, board, piece):
        moves = self.get_possible_moves(board, piece)
        scores = []
        simulated_moves = []
        for x, move_piece in moves:
            temp_board = deepcopy(board)
            temp_board.place(x, temp_board.drop_height(move_piece, x), move_piece)
            scores.append(self.evaluate_board(temp_board))
            simulated_moves.append((x, move_piece))
        if not scores:
            return (0, piece)
        scores = np.array(scores)
        best = np.argmax(scores)
        return simulated_moves[best]

chromosome = np.array([
    -5,
    -2,
    0,
    0,
    100,
    0,
    0,
    0,
    0
])
gene = Genetic_AI(chromosome)
game_instance = Game(mode="genetic", agent=gene)

with open(os.devnull, 'w') as f, contextlib.redirect_stdout(f):
    pieces_dropped, rows_cleared = game_instance.run_no_visual()

print("pieces dropped:", pieces_dropped)
print("rows cleared:", rows_cleared)

pieces dropped: 491
rows cleared: 196


Each individual agent has a fixed chromosome. It evaluates the board position for every possible move according to its chromosome, and then plays the move with the highest evaluation.

In [5]:
survival_rate = 9
mut_rate = 0.5
mut_scale = 5

population = [np.array([-5, -2, 0, 0, 100, 0, 0, 0, 0]) for _ in range(survival_rate)]
generation = 1

def evaluate_fitness(agent):
    n = 3
    total_cleared = 0
    for _ in range(n):
        game = Game(mode="genetic", agent=Genetic_AI(agent))
        rows = game.run_no_visual()[1]
        total_cleared += rows
    return total_cleared / n

def mutate(agent, mut_rate=mut_rate, mut_scale=mut_scale):
    for i in range(len(agent)):
        if np.random.rand() < mut_rate:
            agent[i] += np.random.uniform(-mut_scale, mut_scale)
    return agent

def cross(agent1, agent2):
    return 0.5 * (agent1 + agent2)

def next_generation(population):
    for i in range(survival_rate):
        for j in range(i + 1):
            population.append(mutate(cross(population[i], population[j])))
    return population

def evaluate_population(population):
    global generation
    with open(os.devnull, 'w') as _, contextlib.redirect_stdout(_), contextlib.redirect_stderr(_):
        try:
            with ProcessPoolExecutor() as executor:
                results = list(executor.map(evaluate_fitness, population))
        except KeyboardInterrupt:
            executor.shutdown(wait=True)
            print("training closed")
    paired = list(zip(population, results))
    paired.sort(key=lambda x: x[1], reverse=True)
    survivors = [agent for agent, _ in paired[:survival_rate]]
    print(f"{generation}: {[score for _, score in paired[:survival_rate]]}")
    with open("genetic_weights.json", "w") as f:
        json.dump(paired[0][0].tolist(), f)
    with open("log.txt", "a") as f:
        f.write(f"gen {generation} A: {[int(score) for _, score in paired[:5]]}\n")
    generation += 1
    return survivors

for i in range(100):
    population = next_generation(population)
    population = evaluate_population(population)

1: [1693.6666666666667, 1400.3333333333333, 1367.3333333333333, 196.33333333333334, 193.66666666666666, 134.33333333333334, 94.33333333333333, 77.33333333333333, 76.33333333333333]


UnboundLocalError: cannot access local variable 'results' where it is not associated with a value

We initialize the population with a set of weights that we found to work well when we were defining the model. Now to get from each generation to the next, we retain the top 9 performing agents while removing the rest, and cross and mutate every pair of (not necessarily distinct) survivors. We make use of elitism to ensure that strong algorithms are not lost, but in the next generation the majority of agents have been modified, ensuring that we do have sufficient diversity.

# 6. Fine Tune the Model

With the select model descibe the steps taken to acheve the best rusults possiable 


In [ ]:
survival_rate = 15
mut_rate = 0.5
mut_scale = 5

population = [np.zeros(9) for _ in range(survival_rate)]
generation = 1

def evaluate_fitness(agent):
    n = int(min(8, np.floor(np.log2(generation) ** 1.3) + 1))
    total_cleared = 0
    for _ in range(n):
        game = Game(mode="genetic", agent=Genetic_AI(agent))
        rows = game.run_no_visual()[1]
        total_cleared += rows
    return total_cleared / n

def mutate(agent, mut_rate=mut_rate, mut_scale=mut_scale):
    for i in range(len(agent)):
        if np.random.rand() < mut_rate:
            agent[i] += np.random.uniform(-mut_scale, mut_scale)
    return agent

def cross(agent1, agent2):
    return 0.5 * (agent1 + agent2)

def next_generation(population):
    pairs = [(i, j) for i in range(survival_rate) for j in range(i + 1)]
    pairs = random.sample(pairs, k=survival_rate * 6)
    for (i, j) in pairs:
        population.append(mutate(cross(population[i], population[j])))
    return population

def evaluate_population(population):
    global generation
    with open(os.devnull, 'w') as f, contextlib.redirect_stdout(f):
        with ProcessPoolExecutor() as executor:
            try:
                results = list(executor.map(evaluate_fitness, population))
            except KeyboardInterrupt:
                executor.shutdown(wait=True)
                print("training closed")
    paired = list(zip(population, results))
    paired.sort(key=lambda x: x[1], reverse=True)
    survivors = [agent for agent, _ in paired[:survival_rate]]
    print(f"{generation}: {[score for _, score in paired[:survival_rate]]}")
    with open("genetic_weights.json", "w") as f:
        json.dump(paired[0][0].tolist(), f)
    with open("log.txt", "a") as f:
        f.write(f"gen {generation} B: {[int(score) for _, score in paired[:5]]}\n")
    generation += 1
    return survivors

for i in range(1000):
    population = next_generation(population)
    population = evaluate_population(population)

To fine tune the model, we increase the population size, increase the number of trials each agent gets in order to get more accurate measurements, and let it train overnight.

# 7. Present
In a customer faceing Document provide summery of finding and detail approach taken


Our approach was to use a genetic training model to play tetris. We identified key features in order to quantify how strong a move would be, which gives each agent a way to, in terms of its weights, play what it thinks the optimal move is. With these heuristics alone, we guessed weights that produced an agent which could occasionally clear around 100 lines. We then used the genetic training algorithm to actually train the agents to produce optimal weights. Fairly consistently training increased the strength of the agents, and produced agentsOur training algorithm was mostly successful.

# 8. Launch the Model System
Define your production run code, This should be self susficent and require only your model pramaters 


In [ ]:
import pandas as pd
import cv2
import string
import sys
import os
import numpy as np
import contextlib
from copy import deepcopy
from concurrent.futures import ProcessPoolExecutor
import json

sys.path.append("./tetris_a/src")
from game import Game

def column_heights(grid):
    return np.where(grid.any(axis=0), grid.shape[0] - np.argmax(grid[::-1, :], axis=0), 0)

def total_height(grid, column_heights):
    return np.sum(column_heights)

def count_holes(grid):
    return np.sum((np.cumsum(grid, axis=0) > 0) & (~grid))

def maximum_height(grid, column_heights):
    return np.max(column_heights)

def bumpiness(grid, column_heights):
    return np.sum(np.abs(np.diff(column_heights)))

def wells(grid, column_heights):
    h = grid.shape[0]
    padded = np.concatenate([[h], column_heights, [h]])
    return np.sum(np.maximum(0, np.minimum(padded[:-2], padded[2:]) - padded[1:-1]))

def row_transitions(grid):
    padded = np.pad(grid, ((0, 0), (1, 1)), constant_values=1)
    return np.sum(np.sum(padded[:, :-1] != padded[:, 1:], axis=1))

def column_transitions(grid):
    padded = np.pad(grid, ((0, 1), (0, 0)), constant_values=1)
    return np.sum(np.sum(padded[:-1, :] != padded[1:, :], axis=0))

def total_filled(grid):
    return np.sum(grid)

def valuation(board, chromosome):
    sim_board = deepcopy(board)
    rows_cleared = sim_board.clear_rows()
    grid = np.array(sim_board.board, dtype=bool)
    heights = column_heights(grid)
    features = np.array([
        total_height(grid, heights),
        maximum_height(grid, heights),
        count_holes(grid),
        bumpiness(grid, heights),
        rows_cleared,
        wells(grid, heights),
        row_transitions(grid),
        column_transitions(grid),
        total_filled(grid)
    ])
    return np.dot(features, chromosome)

class Genetic_AI:
    def __init__(self, chromosome=None):
        if chromosome is None:
            self.chromosome = np.random.uniform(-1, 1, 4)
        else:
            self.chromosome = np.array(chromosome)
    def evaluate_board(self, board):
        return valuation(board, self.chromosome)
    def get_possible_moves(self, board, piece):
        seen = set()
        rotations = []
        while True:
            body = tuple(sorted(piece.body))
            if body in seen:
                break
            seen.add(body)
            rotations.append(piece)
            piece = piece.get_next_rotation()
        moves_list = []
        for rotation in rotations:
            max_x = board.width - len(rotation.skirt)
            moves_list.extend([(x, rotation) for x in np.arange(max_x + 1)])
        moves = np.array(moves_list, dtype=object)
        return moves
    def get_best_move(self, board, piece):
        moves = self.get_possible_moves(board, piece)
        scores = []
        simulated_moves = []
        for x, move_piece in moves:
            temp_board = deepcopy(board)
            temp_board.place(x, temp_board.drop_height(move_piece, x), move_piece)
            scores.append(self.evaluate_board(temp_board))
            simulated_moves.append((x, move_piece))
        if not scores:
            return (0, piece)
        scores = np.array(scores)
        best = np.argmax(scores)
        return simulated_moves[best]

chromosome = np.array([
    -5,
    -2,
    0,
    0,
    100.0,
    0,
    0,
    0,
    0
])

with open("genetic_weights.json", "r") as f:
    weights = json.load(f)
model = Genetic_AI(weights)
game_instance = Game(mode="genetic", agent=model)
pieces_dropped, rows_cleared = game_instance.run_no_visual()
print("pieces dropped:", pieces_dropped)
print("rows cleared:", rows_cleared)

pygame 2.6.1 (SDL 2.28.4, Python 3.12.6)
Hello from the pygame community. https://www.pygame.org/contribute.html
1 0
2 0
3 0
4 0
5 0
6 0
7 0
8 0
9 1
10 2
11 2
12 3
13 3
14 3
15 3
16 3
17 3
18 4
19 4
20 4
21 5
22 6
23 7
24 8
25 8
26 8
27 8
28 8
29 8
30 8
31 9
32 9
33 10
34 10
35 10
36 10
37 11
38 11
39 11
40 11
41 11
42 12
43 12
44 12
45 12
46 13
47 13
48 14
49 14
50 15
51 15
52 15
53 16
54 16
55 17
56 17
57 17
58 17
59 18
60 18
61 19
62 19
63 19
64 20
65 20
66 20
67 20
68 21
69 22
70 23
71 24
72 24
73 24
74 24
75 25
76 25
77 25
78 26
79 26
80 26
81 27
82 27
83 27
84 27
85 29
86 30
87 31
